# Mulai di sini — IoTCom.Net dalam 10 menit

IoTCom.Net berbicara protokol industri, navigasi, pencahayaan, dan pesan dengan satu API yang konsisten: **client** terhubung, **server** melayani, **publisher** menerbitkan, **subscriber** berlangganan. Setiap protokol punya simulator, jadi setiap sel di sini berjalan tanpa perangkat keras.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.15.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.15.0-preview.1"

## Bentuk setiap endpoint

Semua endpoint memiliki `State` dan `StateChanged`, merupakan `IAsyncDisposable`, dan dikonfigurasi dengan builder fluent. Transport bisa ditukar: `UseTcp`, `UseSerial`, `UseInMemory`.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Modbus;
using IoTCom.Net.Transports;

var link = new InMemoryTransportListener();
var server = ModbusServer.Create(o => o.ListenInMemory(link));
server.StateChanged += (_, e) => Console.WriteLine($"server: {e.Previous} -> {e.Current}");
await server.StartAsync();

var client = ModbusClient.Create(o => o.UseInMemory(link));
server.Store.HoldingRegisters[0] = 42;
Console.WriteLine($"HR0 = {(await client.ReadHoldingRegistersAsync(0, 1))[0]}");

## Lihat byte-nya

Pasang `RecordingTap` ke endpoint mana pun untuk menangkap setiap frame — data yang sama ditampilkan Galeri dan CLI sebagai *frame lane*.

In [ ]:
var tap = new RecordingTap();
client.AddTap(tap);
await client.ReadHoldingRegistersAsync(0, 2);
foreach (var f in tap.Snapshot()) Console.WriteLine($"{f.Direction,-8} {HexDump.ToHex(f.Data.Span),-40} {f.Summary}");

## Selanjutnya

| Notebook | Topik |
|---|---|
| `industrial/01-modbus` | Master, slave, simulator Modbus, mesin Rust |
| `transport/02-framing-crc` | Katalog CRC, SLIP, COBS, HDLC |
| `navigation/03-nmea` | GPS/GNSS dengan NMEA 0183 |
| `messaging/04-mqtt-senml` | Pub/sub MQTT dengan payload SenML |
| `messaging/12-mdns-sparkplug` | Penemuan mDNS, Sparkplug B, Protobuf/MessagePack/TLV |
| `industrial/13-opcua` | OPC UA: jelajah, baca, subscribe, tulis |
| `devices/14-ble` | Bluetooth LE: advertisement, GATT, notifikasi |
| `devices/15-usb` | Transfer USB control/bulk dan report HID |
| `99-protocol-chooser` | Protokol mana untuk tugas apa |

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*